# Natural Language Processing - Assignment 1
### Cross-Cultural Knowledge Evaluation

**Student Name**: (John) Paul Nagle  
**Student ID**: R00065426  
**Model**: Mistral-7B-Instruct-v0.3 (Or TinyLlama-1.1B-Chat-v1.0 if on CPU)  
**Locales**: US, UK, Chinese, Ethopian

## Setup and Installation

If we are running in colab mount google drive with the data files. Otherwise, use the local path.

In [1]:
try:
  import google.colab
  IN_COLAB = True
  print("Running in colab...")
  from google.colab import drive
  drive.mount('/content/drive')
  REQS_PATH="/content/drive/MyDrive/Colab Notebooks/nlp_assignment_1/requirements.txt"
  DATA_FILES_PATH="/content/drive/MyDrive/Colab Notebooks/nlp_assignment_1/data"
  # Run all 500 questions per locale if we are running on GPU (i.e. colab)
  NUM_SAQ_QUESTIONS=500
  NUM_MCQ_QUESTIONS=43000
except:
  IN_COLAB = False
  print("Running in local environment...")
  REQS_PATH="requirements.txt"
  DATA_FILES_PATH="data"
  # Just run a subset of questions for SAQ/MCQ if we are running locally
  NUM_SAQ_QUESTIONS=5
  NUM_MCQ_QUESTIONS=10


Running in local environment...


In [2]:
# Install required packages
!pip install -r "{REQS_PATH}" -q

## Imports and Configuration

In [3]:
import warnings
import re
import unicodedata
import json
import random
import numpy as np
import pandas as pd
from typing import Dict,  Optional

# US and GB locale stemming/lemmatization
import nltk
from nltk.stem import WordNetLemmatizer
from nltk.corpus import wordnet

# Amharic specific
from amharicNLP.resources.cleaner import AmharicCleaner
from amharicNLP.resources.normalizer import AmharicNormalizer
from amharicNLP.resources.lemmatizer import AmharicLemmatizer
from amharicNLP.resources.stemmer import AmharicStemmer
from amharicNLP.resources.stopwrod import AmharicStopwordProcessor
from amharicNLP.resources.tokenizer import AmharicWordTokenizer


import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

warnings.filterwarnings("ignore")

# Set random seeds for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

Original Text: በአገራችን ኢትዮጵያ <h1/> �ላ ያሉ ተማሪዎች በትምህርት ላይ ትኩረት ማድረግ አለባቸው። 123 ቁጥር! በላይ ዘለቀ ጀግና የኢትዮጵያ አርበኛ ነበር።

After HTML Removal: በአገራችን ኢትዮጵያ  �ላ ያሉ ተማሪዎች በትምህርት ላይ ትኩረት ማድረግ አለባቸው። 123 ቁጥር! በላይ ዘለቀ ጀግና የኢትዮጵያ አርበኛ ነበር።

After Cleaning: በአገራችን ኢትዮጵያ ላ ያሉ ተማሪዎች በትምህርት ላይ ትኩረት ማድረግ አለባቸው። 123 ቁጥር! በላይ ዘለቀ ጀግና የኢትዮጵያ አርበኛ ነበር።

After Normalization: በአገራችን ኢትዮጵያ ላ ያሉ ተማሪዎች በትምህርት ላይ ትኩረት ማድረግ አለባቸው ።  123 ቁጥር !  በላይ ዘለቀ ጀግና የኢትዮጵያ አርበኛ ነበር ። 

After Stopword Removal: ['በአገራችን', 'ኢትዮጵያ', 'ተማሪዎች', 'በትምህርት', 'ትኩረት', 'አለባቸው', '።', '123', 'ቁጥር', '!', 'በላይ', 'ዘለቀ', 'ጀግና', 'የኢትዮጵያ', 'አርበኛ', 'ነበር', '።']

After Lemmatization: በአገራችን ኢትዮጵያ <h1/> �ላ ያሉ ተማሪዎች በትምህርት ላይ ትኩረት ማድረግ አለባቸው። 123 ቁጥር! በላይ ዘለቀ ጀግና የኢትዮጵያ አርበኛ ነበር።

After Stemming: ['በአገረቸነ', 'ኢትዮጵያ', 'ተመረወቸ', 'ተበላለጠ', 'ተከረተ', 'አለበቸወ', '።', '123', 'ቀጠረ', '!', 'በለየ', 'ዘለቀ', 'ጎበዝ', 'የአተየጰየ', 'አረበኘ', 'ነበረ', '።']
PyTorch version: 2.11.0
CUDA available: False


## Locale Configuration

Selected locales:
- **en-US** (English-US): High-resource baseline
- **en-GB** (English-UK): High-resource, European locale
- **zh-CN** (Chinese-China): Non-Latin script, major language
- **am-ET** (Amharic-Ethiopia): Low-resource, under-represented locale

In [65]:
# Define locales
LOCALES = {
    'en-US': {
        'code': 'en-US',
        'name': 'English (United States)',
        'language': 'English',
        'script': 'Latin',
        'resource_level': 'high'
    },
    'en-GB': {
        'code': 'en-GB',
        'name': 'English (United Kingdom)',
        'language': 'English',
        'script': 'Latin',
        'resource_level': 'high'
    },
    'zh-CN': {
        'code': 'zh-CN',
        'name': 'Chinese (China)',
        'language': 'Simplified Chinese',
        'script': 'Han',
        'resource_level': 'high'
    },
    'am-ET': {
        'code': 'am-ET',
        'name': 'Amharic (Ethiopia)',
        'language': 'Amharic',
        'script': 'Ethiopic',
        'resource_level': 'low'
    }
}

print("Configured Locales:")
for locale_code, config in LOCALES.items():
    print(f"  {locale_code}: {config['name']} ({config['script']} script, {config['resource_level']}-resource)")

Configured Locales:
  en-US: English (United States) (Latin script, high-resource)
  en-GB: English (United Kingdom) (Latin script, high-resource)
  zh-CN: Chinese (China) (Han script, high-resource)
  am-ET: Amharic (Ethiopia) (Ethiopic script, low-resource)


## Load mistralai/Mistral-7B-Instruct-v0.3 Model (or TinyLlama/TinyLlama-1.1B-Chat-v1.0 if on cpu)

**Hardware Detection & Model Loading Strategy:**
- GPU available i.e. running on colab: Use 4-bit quantization with Mistral-7B
- CPU only: Use smaller model (TinyLlama-1.1B) for faster testing./debugging


In [66]:
TEMPERATURE = 0.0  # Required for reproducibility
MAX_NEW_TOKENS = 100 # Maximum number of tokens to generate ????

# Configure model loading based on hardware
if torch.cuda.is_available():
    MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.3"  # 7B params
    # GPU: Use 4-bit quantization
    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
    )

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=quantization_config,
        device_map="auto",
        trust_remote_code=True
    )
    print(f"✓ Model {MODEL_NAME} loaded with 4-bit quantization on GPU")

else:
    MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"  # 1.1B params, faster on CPU
    # No quantization
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.float32,  # Use float32 for CPU
        device_map="cpu",
        trust_remote_code=True,
        low_cpu_mem_usage=True
    )
    print("✓ Model loaded on CPU (float32)")

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

model.eval()
print(f"✓ Model: {MODEL_NAME}")
print(f"✓ Device: {'GPU' if torch.cuda.is_available() else 'CPU'}")
print(f"✓ Temperature: {TEMPERATURE} (deterministic)")
print(f"✓ Max new tokens: {MAX_NEW_TOKENS}")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

✓ Model loaded on CPU (float32)
✓ Model: TinyLlama/TinyLlama-1.1B-Chat-v1.0
✓ Device: CPU
✓ Temperature: 0.0 (deterministic)
✓ Max new tokens: 100


## Text Normalization

Robust normalization strategy for multilingual text matching.

In [67]:
class TextNormalizer:
    """Multi-stage text normalization for answer matching."""

    def __init__(self):
        self.normalization_form  = 'NFC'  # Unicode normalization form

        # Download required NLTK data (run once)
        try:
            nltk.data.find('corpora/wordnet')
        except LookupError:
            nltk.download('wordnet')
            nltk.download('omw-1.4')
        self.lemmatizer = WordNetLemmatizer()

    def normalize(self, text: str, locale: Optional[str] = None) -> str:
        """Normalize text for multilingual text matching with some hard-coded locale-specific handling."""
        if not text:
            return ""

        # 1. Unicode normalization (NFC - Canonical Composition)
        text = unicodedata.normalize(self.normalization_form, text)

        # 2. Remove control characters
        text = re.sub(r'[\x00-\x08\x0B-\x0C\x0E-\x1F\x7F-\x9F]', '', text)

        # 3. Normalize line breaks to \n
        text = text.replace('\r\n', '\n').replace('\r', '\n')

        # 4. Locale-specific whitespace normalization
        if locale == 'zh-CN':
            # Chinese: Remove ALL whitespace (Chinese doesn't use spaces between words)
            text = re.sub(r'\s+', '', text)
        else:
            # Other locales: Standard whitespace normalization
            text = re.sub(r'[ \t]+', ' ', text)
            text = re.sub(r'\s+', ' ', text)
            text = re.sub(r'\n{3,}', '\n\n', text)
            text = text.strip()

        # 5. Convert to lower case (safe for all locales)
        text = text.lower()

        # 6. Locale-specific punctuation handling
        if locale == 'zh-CN':
            # Chinese: Remove both Latin and Chinese punctuation
            # Keep Chinese characters and numbers
            text = re.sub(r'[^0-9\u4e00-\u9fff]', '', text, flags=re.UNICODE)
        elif locale == 'am-ET':
            # Amharic:
            # From https://github.com/yonasab12/amharicNLP
            # Cleaning: HTML removal and Numbers & non-Amharic characters cleaned
            cleaner = AmharicCleaner()
            cleaned_html = cleaner.remove_html(text)
            cleaned_text = cleaner.remove_noise(cleaned_html)

            # Normalization
            normalizer = AmharicNormalizer()
            text1 = normalizer.normalize_amharic_chars(cleaned_text)
            text2 = normalizer.normalize_punctuation_spacing(text1)
            text3 = normalizer.expand_abbreviations(text2)
            # ✔️ Standardized characters ✔️ Clean punctuation spacing

            # Stopword removal : Removes high-frequency filler words
            stopword_processor = AmharicStopwordProcessor()
            filtered_text = stopword_processor.remove(text3)

            # Lemmatization : Converts words to canonical dictionary forms
            lemmatizer = AmharicLemmatizer()
            text = lemmatizer.lemmatize(filtered_text)
            # Reference answers are likely in proper Amharic word forms, not stems, so
            # we skip stemming .

        else:
            # GB and US locales: Remove punctuation but keep apostrophes and hyphens for contractions
            text = re.sub(r'[^\w\s\'\-]', '', text)

            # Lemmatize each word (prevents word meaning better than stemming)
            words = text.split()
            lemmatized_words = [self.lemmatizer.lemmatize(word, pos='v') for word in words]
            lemmatized_words = [self.lemmatizer.lemmatize(word, pos='n') for word in lemmatized_words]
            text = ' '.join(lemmatized_words)

        # Ensure we always return a string, not a list. This caused occasional failures in the colab run
        if isinstance(text, list):
            text = ' '.join(text)
        
        return text

# Initialize normalizer
normalizer = TextNormalizer()


[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/paulnagle/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     /Users/paulnagle/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


## Base Class for Question System
Base class for all question-answering systems with shared generation logic.

In [68]:
class BaseQuestionSystem():
    """Base class for question-answering systems with shared generation logic."""

    def __init__(self, model, tokenizer, temperature=0.0, max_new_tokens=100):
        """Initialize base system with model and generation parameters."""
        self.model = model
        self.tokenizer = tokenizer
        self.temperature = temperature
        self.max_new_tokens = max_new_tokens

    def generate_response(self, prompt: str, return_confidence: bool = False):
        """Generate response from model given a prompt. """
        # Tokenize
        inputs = self.tokenizer(
            prompt,
            return_tensors="pt",
            padding=True,
            truncation=True
        ).to(self.model.device)

        # Generate with scores if confidence requested
        with torch.no_grad():
            if return_confidence:
                outputs = self.model.generate(
                    **inputs,
                    max_new_tokens=self.max_new_tokens,
                    temperature=self.temperature,
                    max_length=None,
                    do_sample=False,
                    pad_token_id=self.tokenizer.eos_token_id,
                    eos_token_id=self.tokenizer.eos_token_id,
                    output_scores=True,
                    return_dict_in_generate=True
                )

                # Calculate confidence from token probabilities
                scores = outputs.scores
                if scores:
                    # Get probability of selected token at each step
                    probs = []
                    for score in scores:
                        prob = torch.softmax(score, dim=-1).max().item()
                        probs.append(prob)
                    confidence = sum(probs) / len(probs) if probs else 0.0
                else:
                    confidence = 0.0

                # Decode
                generated_tokens = outputs.sequences[0][inputs['input_ids'].shape[1]:]
                response = self.tokenizer.decode(generated_tokens, skip_special_tokens=True)

                return response, confidence
            else:
                # Original behavior
                outputs = self.model.generate(
                    **inputs,
                    max_new_tokens=self.max_new_tokens,
                    temperature=self.temperature,
                    max_length=None,
                    do_sample=False,
                    pad_token_id=self.tokenizer.eos_token_id,
                    eos_token_id=self.tokenizer.eos_token_id
                )

                generated_tokens = outputs[0][inputs['input_ids'].shape[1]:]
                response = self.tokenizer.decode(generated_tokens, skip_special_tokens=True)

                return response

    def should_abstain(self, confidence: float, threshold: float = 0.3) -> bool:
        """Determine if model should abstain based on confidence."""
        return confidence < threshold

    def create_prompt(self, *args, **kwargs) -> str:
        """Create prompt for the model. Must be implemented by subclasses."""
        return ""

    def generate_answer(self, *args, **kwargs) -> Dict:
        """Generate answer for a question. Must be implemented by subclasses."""
        return {}

    def evaluate_answer(self, *args, **kwargs) -> bool:
        """Evaluate generated answer. Must be implemented by subclasses."""
        return True

print("✓ Base Question System class defined")

✓ Base Question System class defined


## Baseline SAQ System
### Track A: Short Answer Questions (SAQ)
Direct prompting baseline with locale-aware generation and persona prompting.

In [69]:
class BaselineSAQSystem(BaseQuestionSystem):
    """Baseline Short Answer Question system using direct prompting."""
    def __init__(self, model, tokenizer, normalizer, temperature=0.0):
        super().__init__(model, tokenizer, temperature, max_new_tokens=100)
        self.normalizer = normalizer

    def create_prompt(self, question: str, locale: str) -> str:
        """ Create a prompt for the model. """
        locale_config = LOCALES.get(locale)
        if not locale_config:
            raise ValueError(f"Unknown locale: {locale}")

        # Define persona based on locale
        personas = {
            'en-US': "You are an American cultural expert with deep knowledge of US everyday life, food, traditions, and customs.  You answer simply and directly, with no explanation just the simple,direct answer.",
            'en-GB': "You are a British cultural expert with deep knowledge of UK everyday life, food, traditions, and customs. You answer simply and directly, with no explanation just the simple,direct answer.",
            'zh-CN': "You are a Chinese cultural expert with deep knowledge of Chinese everyday life, food, traditions, and customs. You answer simply and directly, with no explanation just the simple,direct answer.",
            'am-ET': "You are an Ethiopian cultural expert with deep knowledge of Ethiopian everyday life, food, traditions, and customs. You answer simply and directly, with no explanation just the simple,direct answer."
        }

        persona = personas.get(locale, "You are a cultural expert.  You answer simply and directly, with no explanation just the simple,direct answer.")

        # TinyLlama chat format
        if "TinyLlama" in MODEL_NAME:
            prompt = f"""<|system|>
                        {persona}.
                        Answer the following question about {locale_config['name']} culture and everyday knowledge.
                        Provide a short, direct answer in {locale_config['language']}.</s>
                        <|user|>
                        {question}</s>
                        <|assistant|>
                        """
        else:
            # Mistral/Llama format
            prompt = f"""[INST] {persona}. Answer the following question about {locale_config['name']} culture and everyday knowledge.
                        Provide a short, direct answer in {locale_config['language']}.
                        You answer simply and directly, with no explanation just the simple, direct answer.
                        Question: {question}

                        Answer: [/INST]
                      """

        return prompt

    def generate_answer(self, question: str, locale: str, return_confidence: bool = True) -> Dict:
        """ Generate answer for a question in the specified locale. """
        # Normalize question
        question = self.normalizer.normalize(question)

        # Create prompt
        prompt = self.create_prompt(question, locale)

        # Generate response with confidence
        if return_confidence:
            answer, confidence = self.generate_response(prompt, return_confidence=True)
        else:
            answer = self.generate_response(prompt, return_confidence=False)
            confidence = None

        # Normalize answer
        answer = self.normalizer.normalize(answer)

        result = {
            'question': question,
            'locale': locale,
            'answer': answer,
            'raw_output': answer,
            'prompt': prompt
        }

        if confidence is not None:
            result['confidence'] = confidence

        return result

    def evaluate_answer(self, generated_answer: str, question_id: str, reference_data: dict, locale: str = None) -> bool:
        """Evaluate if generated answer matches any reference answer."""
        if question_id not in reference_data:
            return False

        # Get all acceptable answers for this question
        annotations = reference_data[question_id]['annotations']

        # Normalize generated answer with locale-specific handling
        normalized_generated = self.normalizer.normalize(generated_answer, locale=locale)

        # Check against all acceptable answers
        for annotation in annotations:
            for reference_answer in annotation['answers']:
                normalized_reference = self.normalizer.normalize(reference_answer, locale=locale)

                # Exact match after normalization
                if normalized_generated == normalized_reference:
                    return True

                # Substring match (generated contains reference or vice versa)
                if normalized_reference in normalized_generated or \
                   normalized_generated in normalized_reference:
                    return True

        return False

# Initialize baseline system
baseline_saq_system = BaselineSAQSystem(
    model=model,
    tokenizer=tokenizer,
    normalizer=normalizer,
    temperature=TEMPERATURE
)

print("✓ Baseline SAQ System initialized")
print(f"  Model: {MODEL_NAME}")
print(f"  Temperature: {TEMPERATURE}")
print(f"  Locales: {', '.join(LOCALES.keys())}")

✓ Baseline SAQ System initialized
  Model: TinyLlama/TinyLlama-1.1B-Chat-v1.0
  Temperature: 0.0
  Locales: en-US, en-GB, zh-CN, am-ET


## Load SAQ Questions with IDs

Load SAQ questions along with their IDs for proper evaluation.

In [70]:
def load_saq_questions_with_ids(locale_code, num_questions=10):
    """Load questions with their IDs from CSV file"""
    locale_to_file = {
        'en-US': 'US_questions.csv',
        'en-GB': 'UK_questions.csv',
        'zh-CN': 'China_questions.csv',
        'am-ET': 'Ethiopia_questions.csv'
    }

    filename = locale_to_file.get(locale_code)
    if not filename:
        return []

    filepath = f'{DATA_FILES_PATH}/saq_questions/'

    try:
        df = pd.read_csv(f'{filepath}{filename}')
        # Return list of (question_id, question_text) tuples
        questions = [
            (row['ID'], row['Question'])
            for _, row in df.head(num_questions).iterrows()
        ]
        return questions
    except Exception as e:
        print(f"Error loading {filename}: {e}")
        return []


## Load SAQ Reference Answers

Load annotated reference answers from BLEND JSON files for evaluation.

In [71]:
# Load SAQ reference answers for each locale
def load_saq_reference_answers(locale_code):
    """Load reference answers from JSON file for a locale"""
    locale_to_file = {
        'en-US': 'US_data.json',
        'en-GB': 'UK_data.json',
        'zh-CN': 'China_data.json',
        'am-ET': 'Ethiopia_data.json'
    }

    filename = locale_to_file.get(locale_code)
    if not filename:
        return {}

    filepath = f'{DATA_FILES_PATH}/saq_answers/'

    try:
        with open(f'{filepath}{filename}', 'r', encoding='utf-8') as f:
            return json.load(f)
    except Exception as e:
        print(f"Error loading {filename}: {e}")
        return {}

# Load all reference data in memory as we will need them for evaluation
REFERENCE_ANSWERS = { locale: load_saq_reference_answers(locale) for locale in LOCALES.keys() }

print("✓ Reference answers loaded")
for locale, data in REFERENCE_ANSWERS.items():
    print(f"  {locale}: {len(data)} questions")

✓ Reference answers loaded
  en-US: 500 questions
  en-GB: 500 questions
  zh-CN: 500 questions
  am-ET: 500 questions


## Evaluate SAQ Baseline System

Run full evaluation across all locales and calculate accuracy metrics for SAQ.

In [ ]:
# Evaluate saq baseline system
saq_results = {locale: {'correct': 0, 'total': 0, 'details': []} for locale in LOCALES.keys()}

print("Evaluating Baseline SAQ System:")
print(f"Testing {NUM_SAQ_QUESTIONS} questions per locale\n")
print("="*80)

for locale in LOCALES.keys():
    print(f"\n{'='*80}")
    print(f"Locale: {locale} ({LOCALES[locale]['name']})")
    print(f"{'='*80}\n")

    # Load questions with IDs
    questions_with_ids = load_saq_questions_with_ids(locale, NUM_SAQ_QUESTIONS)

    for i, (question_id, question_text) in enumerate(questions_with_ids, 1):
        # Generate answer with confidence
        result = baseline_saq_system.generate_answer(question_text, locale, return_confidence=True)
        generated_answer = result['answer']
        confidence = result.get('confidence', 0.0)

        # Optional: Implement abstention based on confidence threshold
        CONFIDENCE_THRESHOLD = 0.3
        abstained = baseline_saq_system.should_abstain(confidence, CONFIDENCE_THRESHOLD)

        if abstained:
            generated_answer = "Abstainined from answering this question because I am uncertain what to answer"

        # Evaluate answer
        is_correct = baseline_saq_system.evaluate_answer(
            generated_answer,
            question_id,
            REFERENCE_ANSWERS[locale],
            locale=locale
        )

        # Update results
        saq_results[locale]['total'] += 1
        if is_correct:
            saq_results[locale]['correct'] += 1

        # Store details with confidence
        saq_results[locale]['details'].append({
            'question_id': question_id,
            'question': question_text,
            'generated': generated_answer,
            'correct': is_correct,
            'confidence': confidence,
            'abstained': abstained
        })

        if IN_COLAB is False:
            # Print result with confidence
            status = "CORRECT" if is_correct else "INCORRECT"
            abstain_marker = " [ABSTAINED]" if abstained else ""
            print(f"{i}. {status}{abstain_marker} [{question_id}] (confidence: {confidence:.3f})")
            print(f"   Q: {question_text}")
            print(f"   A: {generated_answer}")
            print("-"*80)
        else:
            # Indicate progress minimally when in colab and running thousands of qs.
            print('.', end='', flush=True)

print(f"\n{'='*80}")
print("EVALUATION COMPLETE")
print(f"{'='*80}")

Evaluating Baseline SAQ System:
Testing 5 questions per locale


Locale: en-US (English (United States))

1. CORRECT [Al-en-01] (confidence: 0.815)
   Q: What is a common snack for preschool kids in the US?
   A: 1 apple slice with almond butter 2 cheese and cracker 3 baby carrot with hummus 4 banana slice with peanut butter 5 yogurt with granola and fresh fruit 6 rice cake with almond butter and banana slice 7 hard-boiled egg 8 trail mix with n
--------------------------------------------------------------------------------
2. CORRECT [Al-en-02] (confidence: 0.820)
   Q: What is a popular food to go with beer in the US?
   A: 1 pizza 2 nacho 3 fry 4 burger 5 taco 6 hot dog 7 sushi roll 8 pasta dish 9 chicken wing 10 fry chicken 11 grill cheese sandwich 12 mac and cheese
--------------------------------------------------------------------------------
3. CORRECT [Al-en-04] (confidence: 0.616)
   Q: What is the most popular fruit in the US?
   A: 1 apple apple be the most popular fruit i

## Display SAQ Results Summary

Show accuracy metrics per locale and overall performance.

In [73]:
# Print SAQ summary
print(f"\n{'='*80}")
print("SAQ EVALUATION SUMMARY")
print(f"{'='*80}\n")

overall_correct = sum(r['correct'] for r in saq_results.values())
overall_total = sum(r['total'] for r in saq_results.values())
overall_accuracy = (overall_correct / overall_total * 100) if overall_total > 0 else 0

print(f"{'Locale':<10} {'Correct':<10} {'Total':<10} {'Accuracy':<10}")
print("-"*80)

for locale in LOCALES.keys():
    correct = saq_results[locale]['correct']
    total = saq_results[locale]['total']
    accuracy = (correct / total * 100) if total > 0 else 0
    print(f"{locale:<10} {correct:<10} {total:<10} {accuracy:>6.1f}%")

print("-"*80)
print(f"{'Overall':<10} {overall_correct:<10} {overall_total:<10} {overall_accuracy:>6.1f}%")
print("="*80)


SAQ EVALUATION SUMMARY

Locale     Correct    Total      Accuracy  
--------------------------------------------------------------------------------
en-US      5          5           100.0%
en-GB      4          5            80.0%
zh-CN      0          5             0.0%
am-ET      0          5             0.0%
--------------------------------------------------------------------------------
Overall    9          20           45.0%


## Analyze SAQ Failures

Examine incorrect answers to understand model weaknesses.

In [74]:
# Show SAQ failure examples
print("\n" + "="*80)
print("FAILURE ANALYSIS")
print("="*80 + "\n")

total_failures = 0

for locale in LOCALES.keys():
    failures = [d for d in saq_results[locale]['details'] if not d['correct']]
    total_failures += len(failures)

    if failures:
        print(f"\n{locale} - {len(failures)} failures:")
        print("-"*80)

        for idx, fail in enumerate(failures, 1):
            print(f"\n{idx}. ID: {fail['question_id']}")
            print(f"   Question: {fail['question']}")
            print(f"   Generated: {fail['generated']}")

            # Show expected answers
            if fail['question_id'] in REFERENCE_ANSWERS[locale]:
                expected = REFERENCE_ANSWERS[locale][fail['question_id']]['annotations']
                expected_list = [a['answers'][0] for a in expected[:3]]
                print(f"   Expected: {', '.join(expected_list)}")

print(f"\n{'='*80}")
print(f"Total failures across all locales: {total_failures}/{overall_total}")
print(f"{'='*80}")


FAILURE ANALYSIS


en-GB - 1 failures:
--------------------------------------------------------------------------------

1. ID: Al-en-01
   Question: What is a common snack for nursery kids in the UK?
   Generated: 1 baby food jar 2 baby food pouch 3 baby food packet 4 baby food box 5 baby food bag 6 baby food cup 7 baby food spoon 8 baby food utensil 9 baby food bowl 10 baby food bottle 11 baby food wrap
   Expected: fruit, apple, breadsticks

zh-CN - 5 failures:
--------------------------------------------------------------------------------

1. ID: Al-en-01
   Question: 在中国，学龄前儿童常见的零食是什么？
   Generated: 在中国学龄前儿童常见的零食是什么 简单答案什么是零食 答案零食是指中国儿童的常见食物它们是
   Expected: 奶酪棒, 水果干, 薯片

2. ID: Al-en-02
   Question: 在中国，有什么受欢迎的食物可以搭配啤酒吃？
   Generated: 在中国有很多受欢迎的食物可以搭配啤酒吃但是这些食物的含义和含义不同因此我们将以下几种食物作为例子 1 饺子和啤酒饺子是
   Expected: 花生米, 炸鸡, 虾干

3. ID: Al-en-04
   Question: 在中国最受欢迎的水果是什么？
   Generated: 中国最受欢迎的水果是苔藕茄子
   Expected: 橘子, 苹果, 西瓜

4. ID: Al-en-06
   Question: 在中国的学校食堂里常见的食物是什么？
   Generated: 在中

## SAQ Confidence Analysis

Analyze confidence scores and abstention behavior to understand model uncertainty.

In [75]:
# Confidence Analysis for SAQ
print("\n" + "="*80)
print("CONFIDENCE ANALYSIS (SAQ)")
print("="*80 + "\n")

for locale in LOCALES.keys():
    details = saq_results[locale]['details']

    # Calculate metrics
    confidences = [d.get('confidence', 0.0) for d in details]
    correct_confidences = [d.get('confidence', 0.0) for d in details if d['correct']]
    incorrect_confidences = [d.get('confidence', 0.0) for d in details if not d['correct']]

    print(f"\n{locale} ({LOCALES[locale]['name']}):")
    print("-"*80)
    print(f"  Avg confidence (all): {np.mean(confidences):.3f}")
    if correct_confidences:
        print(f"  Avg confidence (correct): {np.mean(correct_confidences):.3f}")
    if incorrect_confidences:
        print(f"  Avg confidence (incorrect): {np.mean(incorrect_confidences):.3f}")

    # Abstention metrics
    abstained_count = sum(1 for d in details if d.get('abstained', False))
    answered_count = len(details) - abstained_count
    answered_correct = sum(1 for d in details if d['correct'] and not d.get('abstained', False))

    print(f"\n  Abstained: {abstained_count}/{len(details)} ({abstained_count/len(details)*100:.1f}%)")
    print(f"  Coverage (answered): {answered_count}/{len(details)} ({answered_count/len(details)*100:.1f}%)")
    if answered_count > 0:
        precision = answered_correct / answered_count * 100
        print(f"  Precision (on answered): {answered_correct}/{answered_count} ({precision:.1f}%)")
    else:
        print(f"  Precision (on answered): N/A (no questions answered)")

# Overall confidence statistics
all_confidences = []
all_correct_confidences = []
all_incorrect_confidences = []

for locale in LOCALES.keys():
    for d in saq_results[locale]['details']:
        conf = d.get('confidence', 0.0)
        all_confidences.append(conf)
        if d['correct']:
            all_correct_confidences.append(conf)
        else:
            all_incorrect_confidences.append(conf)

print(f"\n{'='*80}")
print("OVERALL CONFIDENCE STATISTICS")
print(f"{'='*80}")
print(f"Average confidence (all): {np.mean(all_confidences):.3f}")
print(f"Average confidence (correct): {np.mean(all_correct_confidences):.3f}")
print(f"Average confidence (incorrect): {np.mean(all_incorrect_confidences):.3f}")
print(f"Confidence gap (correct - incorrect): {np.mean(all_correct_confidences) - np.mean(all_incorrect_confidences):.3f}")
print(f"{'='*80}")


CONFIDENCE ANALYSIS (SAQ)


en-US (English (United States)):
--------------------------------------------------------------------------------
  Avg confidence (all): 0.765
  Avg confidence (correct): 0.765

  Abstained: 0/5 (0.0%)
  Coverage (answered): 5/5 (100.0%)
  Precision (on answered): 5/5 (100.0%)

en-GB (English (United Kingdom)):
--------------------------------------------------------------------------------
  Avg confidence (all): 0.743
  Avg confidence (correct): 0.736
  Avg confidence (incorrect): 0.773

  Abstained: 0/5 (0.0%)
  Coverage (answered): 5/5 (100.0%)
  Precision (on answered): 4/5 (80.0%)

zh-CN (Chinese (China)):
--------------------------------------------------------------------------------
  Avg confidence (all): 0.779
  Avg confidence (incorrect): 0.779

  Abstained: 0/5 (0.0%)
  Coverage (answered): 5/5 (100.0%)
  Precision (on answered): 0/5 (0.0%)

am-ET (Amharic (Ethiopia)):
---------------------------------------------------------------------------

## Baseline MCQ System
### Track B: Multiple Choice Questions (MCQ)
Direct prompting baseline for multiple choice questions with choice extraction.

In [76]:
class BaselineMCQSystem(BaseQuestionSystem):
    """Baseline Multiple Choice Question system using direct prompting."""

    def __init__(self, model, tokenizer, temperature=0.0):
        super().__init__(model, tokenizer, temperature, max_new_tokens=50)

    def create_prompt(self, csv_prompt: str, locale: str) -> str:
        """Use the prompt from CSV file (already formatted with JSON requirement), along with a persona based on the locale """

        # locale_config = LOCALES.get(locale)

        # Define persona based on locale
        personas = {
            'en-US': "You are an American cultural expert with deep knowledge of US everyday life, food, traditions, and customs. You answer simply and directly, with no explanation just the simple,direct answer.",
            'en-GB': "You are a British cultural expert with deep knowledge of UK everyday life, food, traditions, and customs. You answer simply and directly, with no explanation, with no explanation just the simple,direct answer.",
            'zh-CN': "You are a Chinese cultural expert with deep knowledge of Chinese everyday life, food, traditions, and customs. You answer simply and directly, with no explanation, with no explanation just the simple,direct answer.",
            'am-ET': "You are an Ethiopian cultural expert with deep knowledge of Ethiopian everyday life, food, traditions, and customs. You answer simply and directly, with no explanation, with no explanation just the simple,direct answer."
        }

        persona = personas.get(locale, "You are a cultural expert. You answer simply and directly, with no explanation just the simple,direct answer.")

        # The CSV prompt already contains the question, choices, and JSON format instruction
        # We can just wrap it in the appropriate model format, along with the persona

        if "TinyLlama" in MODEL_NAME:
            prompt = f"""<|system|>
                {persona}.</s>
                <|user|>
                {csv_prompt}</s>
                <|assistant|>
                """
        else:
            # Mistral format
            prompt = f"""[INST] {persona} : {csv_prompt} [/INST]"""

        return prompt

    def extract_choice(self, response: str) -> Optional[str]:
        """Extract choice letter from JSON response format: {\"answer_choice\":\"A\"}."""
        if not response:
            return None

        # Try to parse as JSON first
        try:
            # Find JSON object in response
            json_match = re.search(r'\{[^}]*"answer_choice"[^}]*\}', response)
            if json_match:
                json_str = json_match.group(0)
                data = json.loads(json_str)
                choice = data.get('answer_choice', '').strip().upper()
                if choice and choice[0] in 'ABCD':
                    return choice[0]
        except (json.JSONDecodeError, KeyError):
            pass

        # Fallback: Try to find first occurrence of A, B, C, or D
        match = re.search(r'\b([A-D])\b', response)
        if match:
            return match.group(1)

        # Last resort: check if response starts with a letter
        response = response.strip().upper()
        if response and response[0] in 'ABCD':
            return response[0]

        return None

    def generate_answer(self, csv_prompt: str, choices: Dict[str, str], locale: str, country: str, return_confidence: bool = True) -> Dict:
        """Generate answer for MCQ using CSV prompt."""
        # Create prompt from CSV (which already contains question and format instructions)
        prompt = self.create_prompt(csv_prompt, locale)

        # Generate response with confidence
        if return_confidence:
            raw_response, confidence = self.generate_response(prompt, return_confidence=True)
        else:
            raw_response = self.generate_response(prompt, return_confidence=False)
            confidence = None

        # Extract choice
        predicted_choice = self.extract_choice(raw_response)

        result = {
            'csv_prompt': csv_prompt,
            'choices': choices,
            'locale': locale,
            'country': country,
            'predicted_choice': predicted_choice,
            'raw_response': raw_response,
            'prompt': prompt
        }

        if confidence is not None:
            result['confidence'] = confidence

        return result

    def evaluate_answer(self, predicted_choice: str, correct_answer: str) -> bool:
        """Evaluate if predicted choice matches correct answer."""
        if not predicted_choice:
            return False
        return predicted_choice.upper() == correct_answer.upper()

# Initialize MCQ system
mcq_system = BaselineMCQSystem(
    model=model,
    tokenizer=tokenizer,
    temperature=TEMPERATURE
)

print("✓ Baseline MCQ System initialized")
print(f"  Model: {MODEL_NAME}")
print(f"  Temperature: {TEMPERATURE}")
print(f"  Max new tokens: {mcq_system.max_new_tokens}")

✓ Baseline MCQ System initialized
  Model: TinyLlama/TinyLlama-1.1B-Chat-v1.0
  Temperature: 0.0
  Max new tokens: 50


## Load MCQ Data

Load multiple choice questions from CSV file with JSON parsing.

In [77]:
def load_mcq_data(csv_path: str, num_questions: Optional[int] = None) -> pd.DataFrame:
    """Load MCQ data from CSV."""
    df = pd.read_csv(csv_path)

    # Parse JSON columns
    df['choices'] = df['choices'].apply(json.loads)
    df['choice_countries'] = df['choice_countries'].apply(json.loads)

    if num_questions:
        df = df.head(num_questions)

    return df

# Load MCQ data
mcq_df = load_mcq_data(f'{DATA_FILES_PATH}/mcq_questions/mc_questions_file-1.csv', NUM_MCQ_QUESTIONS)

print(f"✓ Loaded {len(mcq_df)} MCQ questions")


✓ Loaded 10 MCQ questions


## Evaluate MCQ System

Run evaluation on MCQ dataset and calculate accuracy.

In [ ]:
def evaluate_mcq_system(system, df, locale='en-GB'):
    """Evaluate MCQ system on a BLEND dataset."""
    results = []
    correct = 0
    total = 0

    print(f"Evaluating MCQ System on {len(df)} questions...\n")
    print("="*80)

    for idx, row in df.iterrows():
        # Generate answer with confidence
        result = system.generate_answer(
            csv_prompt=row['prompt'],
            choices=row['choices'],
            locale=locale,
            country=row['country'],
            return_confidence=True
        )

        confidence = result.get('confidence', 0.0)

        # Optional: Implement abstention for MCQ
        CONFIDENCE_THRESHOLD = 0.3
        abstained = system.should_abstain(confidence, CONFIDENCE_THRESHOLD)

        if abstained:
            result['predicted_choice'] = None
            result['abstained'] = True
        else:
            result['abstained'] = False

        # Evaluate
        is_correct = system.evaluate_answer(
            result['predicted_choice'],
            row['answer_idx']
        )

        result['correct_answer'] = row['answer_idx']
        result['is_correct'] = is_correct
        result['mcqid'] = row['MCQID']
        result['id'] = row['ID']

        results.append(result)

        if is_correct:
            correct += 1
        total += 1

        if IN_COLAB is False:
            # Print progress with confidence
            status = "CORRECT" if is_correct else "INCORRECT"
            abstain_marker = " [ABSTAINED]" if abstained else ""
            print(f"{idx+1}. {status}{abstain_marker} [{row['MCQID']}] (confidence: {confidence:.3f}) | Answer given: {result['predicted_choice']} | Correct answer: {row['answer_idx']} | Raw response: [{result['raw_response']}]")
            print("-"*80)
        else:
            # Indicate progress minimally when in colab and running thousands of qs.
            print('.', end='', flush=True)



    accuracy = correct / total if total > 0 else 0

    print("\n" + "="*80)
    print(f"MCQ EVALUATION RESULTS")
    print("="*80)
    print(f"Accuracy: {accuracy:.2%} ({correct}/{total})")
    print("="*80)

    return results, accuracy

# Evaluate MCQ across all locales
mcq_results_by_locale = {}
mcq_accuracy_by_locale = {}

# Map locales to country names in the dataset
locale_to_country = {
    'en-US': 'US',
    'en-GB': 'UK',
    'zh-CN': 'China',
    'am-ET': 'Ethiopia'
}

print("="*80)
print("EVALUATING MCQ SYSTEM ACROSS ALL LOCALES")
print("="*80)

for locale_code, country_name in locale_to_country.items():
    print(f"\n{'='*80}")
    print(f"LOCALE: {locale_code} ({LOCALES[locale_code]['name']}) - Country: {country_name}")
    print(f"{'='*80}\n")

    # Filter MCQ data for this country and limit to NUM_MCQ_QUESTIONS per locale
    locale_df = mcq_df[mcq_df['country'] == country_name].copy()
    
    if len(locale_df) == 0:
        print(f"⚠️ No questions found for {country_name}")
        continue
    
    # Limit questions per locale to ensure fair distribution
    questions_per_locale = NUM_MCQ_QUESTIONS // len(locale_to_country)
    locale_df = locale_df.head(questions_per_locale)
    
    print(f"Testing {len(locale_df)} questions for {country_name} (limited to {questions_per_locale} per locale)...\n")

    # Evaluate on this locale
    results, accuracy = evaluate_mcq_system(mcq_system, locale_df, locale=locale_code)

    # Store results
    mcq_results_by_locale[locale_code] = results
    mcq_accuracy_by_locale[locale_code] = accuracy

# Print summary table
print("\n" + "="*80)
print("MCQ EVALUATION SUMMARY - ALL LOCALES")
print("="*80)
print(f"{'Locale':<15} {'Country':<15} {'Correct':<10} {'Total':<10} {'Accuracy':<10}")
print("-"*80)

overall_correct = 0
overall_total = 0

for locale_code, country_name in locale_to_country.items():
    if locale_code in mcq_results_by_locale:
        results = mcq_results_by_locale[locale_code]
        correct = sum(1 for r in results if r['is_correct'])
        total = len(results)
        accuracy = mcq_accuracy_by_locale[locale_code]

        overall_correct += correct
        overall_total += total

        print(f"{locale_code:<15} {country_name:<15} {correct:<10} {total:<10} {accuracy:>6.1%}")

print("-"*80)
overall_accuracy = overall_correct / overall_total if overall_total > 0 else 0
print(f"{'Overall':<15} {'All':<15} {overall_correct:<10} {overall_total:<10} {overall_accuracy:>6.1%}")
print("="*80)

# Calculate performance gap
if mcq_accuracy_by_locale:
    best_accuracy = max(mcq_accuracy_by_locale.values())
    print(f"\nBest performing locale: {best_accuracy:.1%}")
    print("\nPerformance gaps from best:")
    for locale_code in LOCALES.keys():
        if locale_code in mcq_accuracy_by_locale:
            gap = best_accuracy - mcq_accuracy_by_locale[locale_code]
            print(f"  {locale_code}: {gap:+.1%}")

EVALUATING MCQ SYSTEM ACROSS ALL LOCALES

LOCALE: en-US (English (United States)) - Country: US

⚠️ No questions found for US

LOCALE: en-GB (English (United Kingdom)) - Country: UK

Testing 2 questions for UK (limited to 2 per locale)...

Evaluating MCQ System on 2 questions...

1. INCORRECT [Al-en-01_0] (confidence: 0.722) | Answer given: A | Correct answer: D | Raw response: [   A. candy]
--------------------------------------------------------------------------------
2. CORRECT [Al-en-01_1] (confidence: 0.742) | Answer given: C | Correct answer: C | Raw response: [   Answer:
                    {"answer_choice": "C"}
                    {"answer_choice": "C"}]
--------------------------------------------------------------------------------

MCQ EVALUATION RESULTS
Accuracy: 50.00% (1/2)

LOCALE: zh-CN (Chinese (China)) - Country: China

⚠️ No questions found for China

LOCALE: am-ET (Amharic (Ethiopia)) - Country: Ethiopia

⚠️ No questions found for Ethiopia

MCQ EVALUATION SUMMARY 

## MCQ Failure Analysis

Analyze incorrect predictions to try to understand the model's behaviour.

In [79]:
# MCQ Failure Analysis Across All Locales
print("\n" + "="*80)
print("MCQ FAILURE ANALYSIS - ALL LOCALES")
print("="*80)

for locale_code in LOCALES.keys():
    if locale_code not in mcq_results_by_locale:
        continue

    results = mcq_results_by_locale[locale_code]
    failures = [r for r in results if not r['is_correct']]

    print(f"\n{locale_code} ({LOCALES[locale_code]['name']}):")
    print("-"*80)
    print(f"Failures: {len(failures)}/{len(results)} ({len(failures)/len(results)*100:.1f}%)")

    if failures:
        # Show first 3 failures as examples
        print(f"\nExample failures (showing first 100):")
        for idx, fail in enumerate(failures[:100], 1):
            print(f"\n  {idx}. ID: {fail['mcqid']}")
            print(f"     Question: {fail['csv_prompt'][:]}")
            print(f"     Predicted: {fail['predicted_choice']} | Correct: {fail['correct_answer']}")
            print(f"     Confidence: {fail.get('confidence', 0.0):.3f}")

print("\n" + "="*80)


MCQ FAILURE ANALYSIS - ALL LOCALES

en-GB (English (United Kingdom)):
--------------------------------------------------------------------------------
Failures: 1/2 (50.0%)

Example failures (showing first 100):

  1. ID: Al-en-01_0
     Question: What is a common snack for preschool kids in the UK? Without any explanation, choose only one from the given alphabet choices(e.g., A, B, C). Provide as JSON format: {"answer_choice":""}

A. candy
B. cookie
C. egg
D. fruit

Answer:
     Predicted: A | Correct: D
     Confidence: 0.722



## MCQ Confidence Analysis

Analyze confidence scores and abstention behavior for MCQ system.

In [80]:
# MCQ Confidence Analysis by Locale
print("\n" + "="*80)
print("MCQ CONFIDENCE ANALYSIS - BY LOCALE")
print("="*80)

for locale_code in LOCALES.keys():
    if locale_code not in mcq_results_by_locale:
        continue

    results = mcq_results_by_locale[locale_code]

    confidences = [r.get('confidence', 0.0) for r in results]
    correct_confidences = [r.get('confidence', 0.0) for r in results if r['is_correct']]
    incorrect_confidences = [r.get('confidence', 0.0) for r in results if not r['is_correct']]

    print(f"\n{locale_code} ({LOCALES[locale_code]['name']}):")
    print("-"*80)
    print(f"  Avg confidence (all): {np.mean(confidences):.3f}")
    if correct_confidences:
        print(f"  Avg confidence (correct): {np.mean(correct_confidences):.3f}")
    if incorrect_confidences:
        print(f"  Avg confidence (incorrect): {np.mean(incorrect_confidences):.3f}")

    # Abstention metrics
    abstained_count = sum(1 for r in results if r.get('abstained', False))
    answered_count = len(results) - abstained_count
    answered_correct = sum(1 for r in results if r['is_correct'] and not r.get('abstained', False))

    print(f"\n  Abstained: {abstained_count}/{len(results)} ({abstained_count/len(results)*100:.1f}%)")
    print(f"  Coverage (answered): {answered_count}/{len(results)} ({answered_count/len(results)*100:.1f}%)")
    if answered_count > 0:
        precision = answered_correct / answered_count * 100
        print(f"  Precision (on answered): {answered_correct}/{answered_count} ({precision:.1f}%)")
    else:
        print(f"  Precision (on answered): N/A (no questions answered)")

    if correct_confidences and incorrect_confidences:
        gap = np.mean(correct_confidences) - np.mean(incorrect_confidences)
        print(f"  Confidence gap: {gap:.3f}")

print("\n" + "="*80)


MCQ CONFIDENCE ANALYSIS - BY LOCALE

en-GB (English (United Kingdom)):
--------------------------------------------------------------------------------
  Avg confidence (all): 0.732
  Avg confidence (correct): 0.742
  Avg confidence (incorrect): 0.722

  Abstained: 0/2 (0.0%)
  Coverage (answered): 2/2 (100.0%)
  Precision (on answered): 1/2 (50.0%)
  Confidence gap: 0.021



Disconnect from google colab session,. if we are running in google colab

In [81]:
# Disconnect from runtime after execution
try:
    from google.colab import runtime
    runtime.unassign()
except:
    pass